In [1]:
import numpy as np

In [10]:
class Perceptron:
    """Labels are -1/ +1. One sample at a time, on purpose"""

    def __init__(self, lr=0.01, epochs=20, seed=1):
        self.lr, self.epochs, self.seed = lr, epochs, seed

    def net_input(self, X):
        return X @ self.w_ + self.b_

    def predict(self, X):
        return np.where(self.net_input(X) >= 0.0, 1, -1)

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.w_ = rng.normal(scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.errors_ = []
        for _ in range(self.epochs):
            errors = 0
            for xi, target in zip(X, y):
                if target * self.net_input(xi) <= 0.0:

                    self.w_ += self.lr * target * xi

                    self.b_ += self.lr * target
                    errors += 1

            self.errors_.append(errors)

        return self

def blobs(n=100, gap=4.5, seed=1):
    """Two Gaussian clouds pushed apart: linearly separable."""
    rng = np.random.default_rng(seed)
    X = np.vstack([rng.normal(loc=[-gap/ 2, 0.0], size=(n // 2, 2)),
                   rng.normal(loc=[gap/ 2, 0.0], size=(n // 2, 2))])
    y = np.hstack([-np.ones(n // 2), np.ones(n // 2)])

    return X, y

def xor(n=100, seed=0):
    """Four clouds, label = sign(x1 * x2): no straight line can split it."""
    rng = np.random.default_rng(seed)
    centers = np.array([[1.0, 1.0], [-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0]])
    labels = np.array([1, 1, -1, -1])
    idx = rng.integers(0, 4, size=n)
    X = centers[idx] + rng.normal(scale=0.3, size=(n,2))
    return X, labels[idx].astype(float)

def accuracy(model, X, y):
    return float(np.mean(model.predict(X) == y))



if __name__ == "__main__":
    X, y = blobs()
    p = Perceptron(lr=0.01, epochs=20).fit(X, y)
    print("separable errors/epoch:", p.errors_)
    print("separable accuracy  :", accuracy(p, X, y))
    print("boundary normal w   :", np.round(p.w_, 4), "bias", round(p.b_, 4))

    Xx, yx = xor()
    q = Perceptron(lr=0.01, epochs=20).fit(Xx, yx)
    print("XOR    errors/epochs:", q.errors_)
    print("XOR    accuracy     :", accuracy(q, Xx, yx))


    from sklearn.linear_model import Perceptron as skperceptron
    sk = skperceptron(eta0=0.01, max_iter=20, tol=None, random_state=1).fit(X, y)
    assert accuracy(p, X, y) == 1.0 and sk.score(X, y) == 1.0
    print("both separate the blobs; cos between the two w vectors:",
          round(float(p.w_ @ sk.coef_[0] /
                      (np.linalg.norm(p.w_) * np.linalg.norm(sk.coef_[0]))), 4))

separable errors/epoch: [2, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
separable accuracy  : 1.0
boundary normal w   : [0.0256 0.0084] bias 0.01
XOR    errors/epochs: [42, 30, 42, 36, 34, 37, 37, 34, 39, 37, 37, 35, 30, 36, 41, 33, 38, 33, 39, 40]
XOR    accuracy     : 0.48
both separate the blobs; cos between the two w vectors: 0.9863
